# 06 · Grid, random, Optuna, CV anidada y comparación honesta — Ames Housing

**Módulo 3 · Sesión 8** — Evaluación y selección de modelos

## Objetivos

Cierra el módulo reemplazando, con las herramientas de `05-sesgo-varianza-validacion.md` y
`06-seleccion-hiperparametros.md`, todo lo que las sesiones 6 y 7 dejaron pendiente:

1. Elegir $\lambda$ con **validación cruzada** en vez del split de validación único e
   informal que usó `04-regularizacion-aplicado.ipynb` (sesión 7).
2. Comparar grid search, random search y Optuna sobre un espacio de dos hiperparámetros, y
   **medir** cuánto le cuesta a cada uno llegar a un resultado igual de bueno.
3. Medir el sesgo optimista de reportar el error sobre el mismo CV que eligió el
   hiperparámetro, con **CV anidada**.
4. Formalizar la comparación pareada que `04-pipeline-caracteristicas-aplicado.ipynb`
   (módulo 2) hizo de manera informal, sobre Ridge vs. Lasso.

**Paquetes:** `pandas`, `numpy`, `scikit-learn`, `scipy`, `optuna`.

In [ ]:
import time

import numpy as np
import pandas as pd
import optuna
from scipy.stats import loguniform, uniform
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, Lasso, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    RandomizedSearchCV,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

optuna.logging.set_verbosity(optuna.logging.WARNING)
SEMILLA = 42

## 1. Mismos datos y pipeline de las sesiones 6 y 7

In [ ]:
datos = pd.read_csv("../datos/ames-housing.csv")
numericas = [
    "gr_liv_area", "total_bsmt_sf", "garage_area", "garage_cars",
    "overall_qual", "overall_cond", "year_built", "year_remod_add",
    "lot_area", "full_bath", "bedroom_abvgr",
]
categoricas = ["bldg_type", "house_style", "central_air"]

X = datos[numericas + categoricas]
y = datos["saleprice"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEMILLA)

transformador = ColumnTransformer(
    [
        ("num", Pipeline([("imputar", SimpleImputer(strategy="median")), ("escalar", StandardScaler())]), numericas),
        ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), categoricas),
    ]
)

## 2. Grid search con validación cruzada, reemplazando el split único de la sesión 7

El `kf` con semilla fija (`05-sesgo-varianza-validacion.md`, `06-seleccion-hiperparametros.md`
sección 4) se reutiliza en todas las búsquedas de este notebook, para que sean comparables
entre sí.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=SEMILLA)

pipe_ridge = Pipeline([("prep", transformador), ("reg", Ridge())])
grid_ridge = {"reg__alpha": np.logspace(-2, 3, 20)}

busqueda_ridge = GridSearchCV(pipe_ridge, grid_ridge, cv=kf, scoring="neg_root_mean_squared_error")
busqueda_ridge.fit(X_train, y_train)

print(f"Mejor alpha (Ridge): {busqueda_ridge.best_params_['reg__alpha']:.2f}")
print(f"RMSE de validación cruzada: ${-busqueda_ridge.best_score_:,.0f}")

El $\alpha$ elegido por 5-fold CV (≈7.8) es pequeño — coherente con la curva de validación
de la sesión 7, que ya mostraba el mínimo cerca de $\lambda \to 0$. La diferencia es que
ahora el número viene acompañado del respaldo de 5 pliegues, no de un solo split.

## 3. Grid vs. random vs. Optuna, con dos hiperparámetros

Con un solo hiperparámetro grid search no tiene gran desventaja. El caso interesante es
Elastic Net, con dos ($\alpha$ y `l1_ratio`) — el escenario donde `06-seleccion-hiperparametros.md`
predice que random search y la optimización bayesiana empiezan a ganarle a la rejilla.

In [ ]:
pipe_en = Pipeline([("prep", transformador), ("reg", ElasticNet(max_iter=20000))])

# Grid: 8 x 8 = 64 combinaciones evaluadas todas.
grid_en = {"reg__alpha": np.logspace(-3, 1, 8), "reg__l1_ratio": np.linspace(0.05, 0.95, 8)}
t0 = time.time()
gs_en = GridSearchCV(pipe_en, grid_en, cv=kf, scoring="neg_root_mean_squared_error")
gs_en.fit(X_train, y_train)
t_grid = time.time() - t0

# Random: 25 combinaciones muestreadas de una distribución continua.
dist_en = {"reg__alpha": loguniform(1e-3, 10), "reg__l1_ratio": uniform(0.05, 0.9)}
t0 = time.time()
rs_en = RandomizedSearchCV(
    pipe_en, dist_en, n_iter=25, cv=kf, scoring="neg_root_mean_squared_error", random_state=SEMILLA
)
rs_en.fit(X_train, y_train)
t_random = time.time() - t0

resultados_busqueda = pd.DataFrame(
    [
        {"método": "Grid (64 combos)", "evaluaciones": 64, "tiempo_s": t_grid, "RMSE_cv": -gs_en.best_score_},
        {"método": "Random (25 iter)", "evaluaciones": 25, "tiempo_s": t_random, "RMSE_cv": -rs_en.best_score_},
    ]
)
resultados_busqueda.round(2)

### Optuna: reutiliza el historial para decidir dónde probar

La función objetivo hace su propia validación cruzada con el mismo `kf`, para que la
comparación con grid y random search sea justa.

In [ ]:
def objetivo(trial):
    alpha = trial.suggest_float("alpha", 1e-3, 10, log=True)
    l1_ratio = trial.suggest_float("l1_ratio", 0.05, 0.95)
    modelo = Pipeline(
        [("prep", transformador), ("reg", ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=20000))]
    )
    rmse_por_pliegue = []
    for tr_idx, va_idx in kf.split(X_train):
        modelo.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
        pred = modelo.predict(X_train.iloc[va_idx])
        rmse_por_pliegue.append(mean_squared_error(y_train.iloc[va_idx], pred) ** 0.5)
    return float(np.mean(rmse_por_pliegue))


resultados_optuna = []
for n_trials in [10, 25]:
    t0 = time.time()
    sampler = optuna.samplers.TPESampler(seed=SEMILLA)  # semilla del muestreador: sección 4 de 06-seleccion-hiperparametros.md
    estudio = optuna.create_study(direction="minimize", sampler=sampler)
    estudio.optimize(objetivo, n_trials=n_trials, show_progress_bar=False)
    resultados_optuna.append(
        {
            "método": f"Optuna ({n_trials} trials)",
            "evaluaciones": n_trials,
            "tiempo_s": time.time() - t0,
            "RMSE_cv": estudio.best_value,
        }
    )

pd.concat([resultados_busqueda, pd.DataFrame(resultados_optuna)], ignore_index=True).round(2)

Con **10 evaluaciones**, Optuna ya iguala el resultado de random search con 25. Con 25,
se acerca al óptimo que grid search encontró revisando las 64 combinaciones completas. Ningún
método "hace trampa": todos ajustan y validan exactamente igual, la diferencia está en cómo
deciden qué combinación probar después.

## 4. ¿Cuánto optimismo hay en reportar el error sobre el mismo CV que eligió $\lambda$?

`06-seleccion-hiperparametros.md`, sección 3: elegir el hiperparámetro y reportar el error
de ese mismo proceso de selección es optimista. Se mide comparando el RMSE "ingenuo" de la
sección 2 contra una CV anidada de verdad.

In [ ]:
rmse_ingenuo = -busqueda_ridge.best_score_  # de la seccion 2: mismo CV elige y reporta

errores_externos = []
for tr_idx, va_idx in KFold(n_splits=5, shuffle=True, random_state=SEMILLA).split(X_train):
    X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
    y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]

    cv_interno = KFold(n_splits=5, shuffle=True, random_state=SEMILLA)
    busqueda_interna = GridSearchCV(pipe_ridge, grid_ridge, cv=cv_interno, scoring="neg_root_mean_squared_error")
    busqueda_interna.fit(X_tr, y_tr)  # elige alpha SOLO con datos de este pliegue externo

    pred_externa = busqueda_interna.predict(X_va)  # evalúa en el pliegue que el interno nunca vio
    errores_externos.append(mean_squared_error(y_va, pred_externa) ** 0.5)

rmse_anidado = np.mean(errores_externos)
ee_anidado = np.std(errores_externos) / np.sqrt(len(errores_externos))

print(f"RMSE ingenuo (mismo CV elige y reporta): ${rmse_ingenuo:,.0f}")
print(f"RMSE de CV anidada:                      ${rmse_anidado:,.0f} ± ${ee_anidado:,.0f}")
print(f"Optimismo medido:                        ${rmse_anidado - rmse_ingenuo:,.0f}")

El optimismo medido es pequeño frente al error estándar de la propia estimación anidada —en
este dataset, con $\lambda$ óptimo ya cercano a OLS, no hay mucho margen de fuga—. No
siempre es así: cuantos más hiperparámetros y más pequeño el dataset, mayor la brecha
esperada entre el número "ingenuo" y el real. La CV anidada es la manera de **saber** cuál es
el caso, en vez de asumirlo.

## 5. Comparación pareada: ¿Ridge le gana a Lasso, de verdad?

Se afinan ambos con grid search (cada uno con su propio $\lambda$ óptimo) y se comparan
sobre los **mismos** 10 pliegues — la comparación pareada de `05-sesgo-varianza-validacion.md`.

In [ ]:
busqueda_lasso = GridSearchCV(
    Pipeline([("prep", transformador), ("reg", Lasso(max_iter=20000))]),
    {"reg__alpha": np.logspace(-2, 3, 20)},
    cv=kf,
    scoring="neg_root_mean_squared_error",
)
busqueda_lasso.fit(X_train, y_train)

modelo_ridge = Pipeline([("prep", transformador), ("reg", Ridge(alpha=busqueda_ridge.best_params_["reg__alpha"]))])
modelo_lasso = Pipeline([("prep", transformador), ("reg", Lasso(alpha=busqueda_lasso.best_params_["reg__alpha"], max_iter=20000))])

kf_pareado = KFold(n_splits=10, shuffle=True, random_state=SEMILLA)
errores_ridge, errores_lasso = [], []
for tr_idx, va_idx in kf_pareado.split(X_train):
    X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
    y_tr, y_va = y_train.iloc[tr_idx], y_train.iloc[va_idx]
    modelo_ridge.fit(X_tr, y_tr)
    modelo_lasso.fit(X_tr, y_tr)
    errores_ridge.append(mean_squared_error(y_va, modelo_ridge.predict(X_va)) ** 0.5)
    errores_lasso.append(mean_squared_error(y_va, modelo_lasso.predict(X_va)) ** 0.5)

diferencias = np.array(errores_ridge) - np.array(errores_lasso)
print(f"Ridge (α={busqueda_ridge.best_params_['reg__alpha']:.1f}) vs. Lasso (α={busqueda_lasso.best_params_['reg__alpha']:.1f})")
print(f"Diferencia media (Ridge - Lasso): ${diferencias.mean():,.0f}")
print(f"Error estándar de la diferencia:  ${diferencias.std()/np.sqrt(len(diferencias)):,.0f}")

La diferencia media es mucho menor que su propio error estándar: **no hay evidencia de que
uno le gane al otro** en este dataset. Es la misma conclusión, con la misma forma, que
`04-pipeline-caracteristicas-aplicado.ipynb` (módulo 2) reportó para las variables nuevas:
"la desviación entre pliegues es varias veces mayor que la diferencia". Reportar "Lasso ganó
por \$X" sin este chequeo habría sido, con altísima probabilidad, reportar ruido.

## Resumen del módulo 3

| Sesión | Lo que se resolvió |
|---|---|
| S6 | Regresión lineal y descenso del gradiente, validados contra la solución exacta |
| S7 | Multicolinealidad, VIF, Ridge/Lasso a mano, y su efecto real (o no) sobre la predicción |
| S8 | Sesgo-varianza, k-fold, grid/random/Optuna, CV anidada, comparación pareada de modelos |

Las tres promesas pendientes de los módulos 1 y 2 quedan cumplidas: la variabilidad de una
sola partición se mide con CV en vez de esconderse (sección 2), la incertidumbre de una
métrica y de la diferencia entre modelos se cuantifica en vez de asumirse (secciones 4 y 5),
y la búsqueda de hiperparámetros ya no se hace a ojo con un solo split de validación.